In [1]:
import torch 
from src.utils import data_config_map, DataType
import json
from src.models import NeuralOperatorModel, ModelType
import matplotlib.pyplot as plt


In [2]:
model_name = "test"

with open(f'./models/{model_name}.json') as f:
    d = json.load(f)

data_type = DataType(d["data"])
data_config = data_config_map.get(data_type)

cheby = False if d["model"] == "fno" else True 
pth = data_config.pth if cheby is False else data_config.cheby_pth

data = torch.load(pth, mmap=True)

# model_params = {k: eval(v) for k, v in (arg.split('=') for arg in d["arg"])}
# model = NeuralOperatorModel(ModelType(d["model"]), data=data, **model_params)

from src.models import OPNO

model = torch.load(f"./models/{model_name}.pt", weights_only=False)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"



In [3]:
from src.utils.data import generate_grid


def visualize_output(model, data_type: DataType, a, u):
    print(a.shape, u.shape)

    if data_type == DataType.BURGERS or DataType.BURGERS_PERIODIC:
        out = model(a)

        # Establish figure subplots and titles 
        fig, axs = plt.subplots(1, 3, figsize=(12, 10))
        

        grid = generate_grid(a.shape[1], cheby)

        markevery = 128
        axs[0][0].plot(grid, a, 'o', ls='-', ms=4, markevery=markevery)
        axs[0][1].plot(grid, u, 'o', ls='-', ms=4, markevery=markevery)
        axs[0][2].plot(grid, out, 'o', ls='-', ms=4, markevery=markevery)

        plt.show()
print(model)
visualize_output(model, data_type=data_type, a=data["a"][-1:][:, ::d["subsample"]].to(device), u=data["u"][-1:][:, ::d["subsample"]].to(device))


NeuralOperatorModel(
  (model): OPNO(
    (conv0): PseudoSpectra()
    (conv1): PseudoSpectra()
    (conv2): PseudoSpectra()
    (conv3): PseudoSpectra()
    (convl): PseudoSpectra()
    (w0): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
    (w1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
    (w2): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
    (w3): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
    (fc1): Linear(in_features=64, out_features=128, bias=True)
    (fc2): Linear(in_features=128, out_features=1, bias=True)
  )
)
torch.Size([1, 257, 1]) torch.Size([1, 257, 1])


RuntimeError: Given groups=1, weight of size [64, 64, 1], expected input[1, 63, 257] to have 64 channels, but got 63 channels instead